# DBSCAN: From First Principles

DBSCAN stands for Density-Based Spatial Clustering of Applications with Noise. It groups observations that are connected through sufficiently dense neighborhoods and labels observations outside those groups as noise.

Unlike K-Means, DBSCAN does not require the number of clusters in advance and can find clusters with irregular shapes. Its notion of a cluster depends on one shared distance scale and one density threshold, so it works best when those assumptions fit the data.

## Why DBSCAN Is Useful

K-Means assigns every point to one of $K$ centroid-based groups, so it can struggle with curved shapes and requires $K$ to be selected first. DBSCAN instead grows groups from dense neighborhoods, can label isolated points as noise, and discovers the number of clusters implied by its density settings.

This makes DBSCAN useful for spatial data, anomaly exploration, and datasets where clusters are irregular but have reasonably consistent density. It is not a universal replacement: a single radius and density threshold can fail when clusters have very different densities or when distances are not meaningful.

## Neighborhoods and Density Parameters

DBSCAN has two main parameters: neighborhood radius $\varepsilon$ and `min_samples`. For a point $p$, its $\varepsilon$-neighborhood is:

$$
N_{\varepsilon}(p)=\{q\in D : d(p,q)\leq\varepsilon\}
$$

Here $d$ is the selected distance metric. The radius sets what counts as nearby; `min_samples` sets how many observations must be nearby for the region to count as dense. In many common APIs, including scikit-learn, the point itself is included in the neighborhood count.

A point is a **core point** if $|N_{\varepsilon}(p)|\geq\texttt{min\_samples}$. Increasing $\varepsilon$ generally makes neighborhoods larger, while increasing `min_samples` makes the density requirement stricter. The result depends on both parameters and the chosen distance metric.

## Core Points, Border Points, and Noise

A core point has at least `min_samples` observations in its $\varepsilon$-neighborhood. It can act as a seed that expands a cluster.

A border point has fewer than `min_samples` observations in its own neighborhood, but lies within the neighborhood of a core point. It can belong to a cluster without expanding that cluster.

A noise point is not density-reachable from any core point and is not assigned to a cluster. An implementation may first encounter a point as noise and later reassign it as a border point when a nearby core point expands its cluster.

Density reachability is directional in its local definition: a point can be directly reached from a core point, but a border point cannot expand the cluster. Chains of core points connect neighborhoods into a larger cluster.

## How DBSCAN Expands a Cluster

DBSCAN scans the observations. When it finds an unvisited point, it queries that point's $\varepsilon$-neighborhood:

1. If the neighborhood has fewer than `min_samples` points, mark the point as noise for now.
2. If it is a core point, start a new cluster and place its neighbors in a candidate queue.
3. Visit candidate points. If an unvisited candidate is also core, add its neighbors to the queue so the cluster can grow through connected dense regions.
4. Add candidates that are not already assigned to this or another cluster. A non-core candidate can join as a border point, but it does not expand the queue.
5. Continue until no reachable candidates remain, then scan for another unvisited core point.

The result is a maximal group connected through density-reachable core points, together with its border points. A border point lying within the neighborhoods of two distinct clusters can be assigned according to traversal order, so some ambiguous boundary assignments may vary across implementations or point order.

## Worked Example in One Dimension

Take points $0$, $0.7$, $1.2$, $1.9$, and $5$, with $\varepsilon=1$ and `min_samples=3` (counting each point itself).

| Point | Points in its neighborhood | Count | Type |
|---:|---|---:|---|
| $0$ | $0, 0.7$ | 2 | Border |
| $0.7$ | $0, 0.7, 1.2$ | 3 | Core |
| $1.2$ | $0.7, 1.2, 1.9$ | 3 | Core |
| $1.9$ | $1.2, 1.9$ | 2 | Border |
| $5$ | $5$ | 1 | Noise |

The core points $0.7$ and $1.2$ are within each other's radius, so they connect into one cluster. The two border points attach to that cluster but do not expand it. The isolated point at $5$ is noise. This example shows why a cluster may contain points whose own neighborhoods are not dense enough.

## Sensitivity to $\varepsilon$ and `min_samples`

Use the same points $\{0,0.7,1.2,1.9,5\}$ and compare these settings. Neighborhood counts include the point itself.

| Settings | Result for $0,0.7,1.2,1.9$ | Result for $5$ |
|---|---|---|
| $\varepsilon=1$, `min_samples` $=3$ | One cluster: core points $0.7,1.2$; border points $0,1.9$ | Noise |
| $\varepsilon=0.6$, `min_samples` $=3$ | No point has 3 neighbors, so all four are noise | Noise |
| $\varepsilon=1$, `min_samples` $=2$ | One cluster; all four points are core | Noise |

Reducing $\varepsilon$ breaks the dense connection and removes every core point in this example. Keeping the radius but lowering `min_samples` changes the border points into core points. This illustrates that the parameters affect both cluster membership and the core/border/noise roles; a tiny parameter change can produce a different partition.

## Choosing Parameters

The distance metric and feature representation define what "nearby" means. For numeric features measured on different scales, scale features deliberately before using a distance-based method; otherwise a large-unit feature can dominate. Fit scaling on training data when evaluating generalization, then apply the same transformation to later data.

`min_samples` sets the density threshold. A larger value requires stronger local support and can classify more observations as noise. `eps` sets the radius and is measured in the units of the transformed feature space. A small change in `eps` can merge clusters, split connectivity, or convert many points to noise.

A common diagnostic is a sorted k-distance plot: calculate each point's distance to its `min_samples`-th nearest neighbor, sort the distances, and look for a bend that separates dense-region distances from larger distances. The bend is a heuristic, not a guaranteed optimum. Inspect cluster counts, noise fraction, stability, and domain usefulness across plausible settings. There is no universal parameter pair that works for every dataset.

## Strengths, Limitations, and Evaluation

DBSCAN can find non-spherical clusters, does not require the number of clusters in advance, and explicitly marks observations that are not density-reachable from a cluster. Its main limitation is the assumption that one global density scale is suitable. A single $\varepsilon$ can merge dense groups, split sparse groups, or label a lower-density cluster as noise when densities vary substantially.

Results also depend on the distance metric and feature scaling. In high dimensions, distances can become less discriminative, and neighborhood search can be expensive. With spatial indexes, neighborhood queries can be efficient on suitable low-dimensional data, but dense or high-dimensional inputs can approach quadratic work and memory in common implementations.

Evaluate more than a single score. Inspect cluster shapes, sizes, the fraction labeled noise, and stability under small parameter changes. A silhouette score can be misleading if all noise points are treated as one ordinary cluster; report how noise was handled and consider metrics designed for density-based clustering when appropriate.

DBSCAN is a good choice when clusters are separated by sparse regions and a shared density threshold is plausible. Consider OPTICS or HDBSCAN when density varies across clusters, and consider K-Means or hierarchical clustering when their geometric assumptions and outputs better match the problem.